In [11]:
using DataStructures
using Distributions

mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves),typeof(x0),typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end


function solve_tabu(p, s::TabuState; iteration_limit::Int=100)
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                # move forbidden, do not consider
                continue
            end
            # evaluate move
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move = i_move
                best_move_obj = considered_value
            end
        end
        # no allowed move found
        if best_move == 0
            break
        end
        apply!(s.current, moves[best_move])
        push!(s.tabu_buffer, invert_move(p, moves[best_move]))
        if best_move_obj < s.best_seen_obj
            # best so far, let's remember it
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    return s.best_seen
end


struct KnapsackProblem
    capacity::Int
    weights::Vector{Int}
    profits::Vector{Int}
end

function objective(p::KnapsackProblem, x)
    return -sum(p.profits .* x)
end


function apply!(x, move::Tuple{Symbol,Int})
    if move[1] === :add
        x[move[2]] = true
    else
        x[move[2]] = false
    end
    return x
end

function invert_move(::KnapsackProblem, move::Tuple{Symbol,Int})
    if move[1] === :add
        return (:remove, move[2])
    else
        return (:add, move[2])
    end
end


function possible_moves(p::KnapsackProblem, x::Vector{Bool})
    move_list = Tuple{Symbol,Int}[]
    current_weight = sum(p.weights .* x)
    # add item
    for i in eachindex(x, p.weights)
        if !x[i] && current_weight + p.weights[i] <= p.capacity
            push!(move_list, (:add, i))
        end
    end
    # remove item
    for i in eachindex(x, p.weights)
        if x[i]
            push!(move_list, (:remove, i))
        end
    end
    return move_list
end



ErrorException: invalid redefinition of constant Main.KnapsackProblem

In [12]:

function generate_problem()
    n_items = 100
    profits = rand(DiscreteUniform(10, 1000), n_items)
    weights = rand(DiscreteUniform(10, 100), n_items)
    kp = KnapsackProblem(3000, profits, weights)
end

kp1 = generate_problem()


MethodError: MethodError: Cannot `convert` an object of type Int64 to an object of type Vector{Int64}
The function `convert` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  convert(::Type{T}, !Matched::T) where T
   @ Base Base.jl:126
  convert(::Type{T}, !Matched::AbstractArray) where T<:Array
   @ Base array.jl:618
  convert(::Type{T}, !Matched::T) where T<:AbstractArray
   @ Base abstractarray.jl:16
  ...


In [36]:

function test(kp)
    x0 = fill(false, length(kp.weights))
    st = TabuState(kp, x0; buffer_length=10)
    sol = solve_tabu(kp, st; iteration_limit=1000000)
    println(findall(sol))
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end

test(kp1)

InexactError: InexactError: Bool(2)

In [14]:
struct MultiKnapsackProblem
    capacities::NTuple{3,Int}
    profits::Vector{Int}
    weights::Vector{Int}
end

function objective(p::MultiKnapsackProblem, x::Vector{Int})
    return -sum(p.profits[i] for i in eachindex(x) if x[i] != 0)
end

function is_feasible(p::MultiKnapsackProblem, x::Vector{Int})
    weights_used = (0, 0, 0)
    for i in eachindex(x)
        bag = x[i]
        if bag != 0
            weights_used = Base.setindex(weights_used, weights_used[bag] + p.weights[i], bag)
        end
    end
    all(weights_used[i] <= p.capacities[i] for i in 1:3)
end

function possible_moves(p::MultiKnapsackProblem, x::Vector{Int})
    moves = Tuple{Symbol,Int,Int}[]
    for i in eachindex(x)
        current = x[i]
        for new_bag in 0:3
            if new_bag != current
                # tymczasowa kopia, sprawdzamy czy będzie spełniać ograniczenia
                x_tmp = copy(x)
                x_tmp[i] = new_bag
                if is_feasible(p, x_tmp)
                    push!(moves, (:assign, i, new_bag))
                end
            end
        end
    end
    return moves
end

function apply!(x::Vector{Int}, move::Tuple{Symbol,Int,Int})
    _, idx, new_bag = move
    x[idx] = new_bag
    return x
end

function invert_move(p::MultiKnapsackProblem, move::Tuple{Symbol,Int,Int})
    return move  # odwrócenie może być tożsame w tej wersji
end


invert_move (generic function with 2 methods)

Test 

In [39]:
using DataStructures
using Random
using Distributions
using Dates  # <-- do pomiaru czasu

# Definicja problemu
struct KnapsackProblem
    capacities::Vector{Int}  # trzy plecaki
    weights::Vector{Int}
    profits::Vector{Int}
end

# Funkcja celu
function objective(p::KnapsackProblem, x)
    total = 0
    for (i, assignment) in enumerate(x)
        if assignment != 0
            total += p.profits[i]
        end
    end
    return -total
end

# Sprawdzanie dopuszczalności rozwiązania
function is_feasible(p::KnapsackProblem, x)
    for plecak in 1:3
        total_weight = sum((p.weights[i] for i in eachindex(x) if x[i] == plecak); init=0)
        if total_weight > p.capacities[plecak]
            return false
        end
    end
    return true
end

# Stosowanie ruchu
function apply!(x, move::Tuple{Symbol,Int,Int})
    action, item_idx, new_assignment = move
    if action == :assign
        x[item_idx] = new_assignment
    elseif action == :unassign
        x[item_idx] = 0
    end
    return x
end

# Odwracanie ruchu
function invert_move(p::KnapsackProblem, move::Tuple{Symbol,Int,Int})
    action, item_idx, new_assignment = move
    if action == :assign
        return (:unassign, item_idx, 0)
    elseif action == :unassign
        return (:assign, item_idx, new_assignment)
    end
end

# Możliwe ruchy
function possible_moves(p::KnapsackProblem, x)
    moves = Tuple{Symbol,Int,Int}[]
    for (i, assignment) in enumerate(x)
        if assignment == 0
            for plecak in 1:3
                total_weight = sum((p.weights[j] for j in eachindex(x) if x[j] == plecak); init=0) + p.weights[i]
                if total_weight <= p.capacities[plecak]
                    push!(moves, (:assign, i, plecak))
                end
            end
        else
            push!(moves, (:unassign, i, assignment))
        end
    end
    return moves
end

# Struktura stanu tabu
mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves), typeof(x0), typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end

# Algorytm tabu search
function solve_tabu(p, s::TabuState; iteration_limit::Int=1000)
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move_idx = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                continue
            end
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            if !is_feasible(p, s.considered)
                continue
            end
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move_idx = i_move
                best_move_obj = considered_value
            end
        end
        if best_move_idx == 0
            break
        end
        apply!(s.current, moves[best_move_idx])
        push!(s.tabu_buffer, invert_move(p, moves[best_move_idx]))
        if best_move_obj < s.best_seen_obj
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    return s.best_seen
end

# Generowanie losowego sensownego startowego rozwiązania
function random_start_solution(p::KnapsackProblem)
    x = fill(0, length(p.weights))
    for i in eachindex(p.weights)
        assign_to = rand(0:3)
        if assign_to != 0
            total_weight = sum((p.weights[j] for j in eachindex(x) if x[j] == assign_to); init=0) + p.weights[i]
            if total_weight <= p.capacities[assign_to]
                x[i] = assign_to
            end
        end
    end
    return x
end

# Funkcja testowa dla wielu losowych przypadków
function test(kp; n_cases=10)
    buffer_lengths = [1, 2, 5]
    results_per_buffer = Dict{Int, Vector{Tuple{Float64, Float64}}}()

    for buffer_length in buffer_lengths
        println("\n=== Testy dla buffer_length = $buffer_length ===")
        results = []
        total_value = 0.0
        total_time = 0.0

        for case in 1:n_cases
            x0 = random_start_solution(kp)

            st = TabuState(kp, x0; buffer_length=buffer_length)
            
            start_time = now()
            sol = solve_tabu(kp, st; iteration_limit=1000)
            end_time = now()
            
            elapsed = (end_time - start_time).value / 1e9  # czas w sekundach
            best_value = -st.best_seen_obj

            println("Przypadek $case: Najlepsza wartość = ", best_value, ", czas = ", elapsed, " sekund")
            
            total_value += best_value
            total_time += elapsed
            
            push!(results, (best_value, elapsed))
        end

        avg_value = total_value / n_cases
        avg_time = total_time / n_cases

        println("\nPodsumowanie dla buffer_length=$buffer_length:")
        println("Średnia wartość: ", avg_value)
        println("Średni czas: ", avg_time, " sekund")

        results_per_buffer[buffer_length] = results
    end

    return results_per_buffer
end

# Przykład użycia
weights = rand(1:10, 90)
profits = rand(10:100, 90)
capacities = [30, 40, 50]

kp = KnapsackProblem(capacities, weights, profits)

# Odpalenie testu
results = test(kp, n_cases=10)



=== Testy dla buffer_length = 1 ===
Przypadek 1: Najlepsza wartość = 1498, czas = 1.1e-8 sekund
Przypadek 2: Najlepsza wartość = 1363, czas = 2.2e-8 sekund
Przypadek 3: Najlepsza wartość = 1274, czas = 1.3e-8 sekund
Przypadek 4: Najlepsza wartość = 1321, czas = 1.4e-8 sekund
Przypadek 5: Najlepsza wartość = 1205, czas = 1.3e-8 sekund
Przypadek 6: Najlepsza wartość = 1394, czas = 1.2e-8 sekund
Przypadek 7: Najlepsza wartość = 1410, czas = 1.2e-8 sekund
Przypadek 8: Najlepsza wartość = 1584, czas = 1.9e-8 sekund
Przypadek 9: Najlepsza wartość = 1170, czas = 1.5e-8 sekund
Przypadek 10: Najlepsza wartość = 1339, czas = 1.7e-8 sekund

Podsumowanie dla buffer_length=1:
Średnia wartość: 1355.8
Średni czas: 1.48e-8 sekund

=== Testy dla buffer_length = 2 ===
Przypadek 1: Najlepsza wartość = 1448, czas = 1.3e-8 sekund
Przypadek 2: Najlepsza wartość = 1561, czas = 1.5e-8 sekund
Przypadek 3: Najlepsza wartość = 1373, czas = 1.3e-8 sekund
Przypadek 4: Najlepsza wartość = 1485, czas = 1.3e-8 sekun

Dict{Int64, Vector{Tuple{Float64, Float64}}} with 3 entries:
  5 => [(1162.0, 1.6e-8), (1588.0, 1.3e-8), (1269.0, 1.4e-8), (1461.0, 1.3e-8),…
  2 => [(1448.0, 1.3e-8), (1561.0, 1.5e-8), (1373.0, 1.3e-8), (1485.0, 1.3e-8),…
  1 => [(1498.0, 1.1e-8), (1363.0, 2.2e-8), (1274.0, 1.3e-8), (1321.0, 1.4e-8),…

In [40]:
capacities0 = [619, 360, 984]
profits0 = [830, 823, 288, 56, 726, 439, 534, 194, 387, 749, 127, 846, 165, 868, 143, 419, 750, 283, 505, 435, 828, 946, 315, 559, 462, 228, 342, 400, 371, 576, 356, 807, 346, 591, 991, 311, 878, 104, 318, 347, 985, 345, 354, 939, 333, 46, 873, 640, 947, 341, 746, 806, 609, 833, 731, 922, 404, 949, 916, 138, 715, 373, 997, 57, 659, 887, 797, 854, 537, 342, 412, 671, 406, 275, 846, 570, 142, 587, 734, 266]
weights0 = [61, 75, 95, 19, 77, 20, 75, 12, 49, 90, 24, 48, 84, 100, 33, 19, 20, 41, 100, 38, 64, 91, 16, 35, 45, 47, 45, 83, 98, 41, 13, 71, 46, 68, 59, 62, 19, 79, 48, 59, 100, 17, 31, 45, 40, 75, 57, 83, 57, 79, 35, 69, 40, 94, 39, 65, 11, 56, 63, 21, 22, 22, 59, 79, 13, 10, 72, 59, 52, 59, 52, 27, 92, 55, 44, 42, 64, 32, 85, 68]

kp0 = KnapsackProblem(capacities, weights, profits)
results0 = test(kp0, n_cases=10)

capacities1 = [985, 534, 988]
profits1 = [675, 262, 976, 36, 613];
weights1 = [20, 34, 65, 11, 26];

kp1 = KnapsackProblem(capacities, weights, profits)
results1 = test(kp1, n_cases=10)

capacities2 = [919, 873, 624];
profits2 = [541, 542, 847, 338, 306, 729, 25, 368, 864, 461];
weights2 = [78, 84, 48, 71, 40, 90, 31, 45, 44, 32];


kp2 = KnapsackProblem(capacities2, weights2, profits2)
results2 = test(kp2, n_cases=10)

capacities3 = [661, 880, 553];
profits3 = [598, 680, 243, 391, 330, 531, 556, 452, 378, 885, 332, 758, 249, 539, 100, 344, 663, 442, 425, 595];
weights3 = [96, 94, 82, 87, 57, 25, 54, 37, 61, 98, 68, 48, 75, 93, 53, 97, 65, 20, 70, 90];

kp3 = KnapsackProblem(capacities3, weights3, profits3)
results3 = test(kp3, n_cases=10)

capacities4 = [256, 281, 766];
profits4 = [983, 938, 671, 69, 703, 664, 700, 204, 356, 41, 270, 820, 76, 832, 658, 290, 475, 187, 36, 869, 124, 106, 949, 977, 113];
weights4 = [89, 10, 30, 35, 37, 65, 37, 31, 43, 68, 47, 43, 87, 42, 31, 67, 29, 15, 79, 75, 88, 41, 34, 22, 23];

kp4 = KnapsackProblem(capacities4, weights4, profits4)
results4 = test(kp4, n_cases=10)

capacities5 = [671, 944, 851];
profits5 = [318, 684, 958, 26, 897, 154, 333, 318, 929, 86, 354, 503, 965, 69, 336, 288, 946, 31, 356, 345, 775, 413, 940, 239, 154, 207, 998, 850, 45, 506];
weights5 = [19, 16, 67, 82, 56, 80, 60, 55, 63, 92, 43, 70, 100, 11, 97, 16, 81, 21, 54, 35, 40, 60, 52, 23, 24, 38, 90, 38, 28, 39];

kp5 = KnapsackProblem(capacities5, weights5, profits5)
results5 = test(kp5, n_cases=10)

capacities6 = [658, 890, 161];
profits6 = [485, 663, 142, 148, 547, 550, 56, 166, 22, 47, 550, 403, 579, 687, 64, 467, 908, 500, 730, 662, 461, 125, 850, 408, 483, 879, 812, 327, 100, 955, 212, 171, 234, 504, 790, 440, 294, 563, 754, 466];
weights6 = [22, 28, 61, 40, 69, 52, 72, 89, 55, 79, 88, 84, 95, 90, 51, 53, 19, 66, 50, 80, 98, 16, 92, 10, 95, 85, 98, 26, 71, 55, 33, 76, 76, 90, 78, 84, 82, 79, 86, 98];

kp6 = KnapsackProblem(capacities6, weights6, profits6)
results6 = test(kp6, n_cases=10)

capacities7 = [325, 302, 604];
profits7 = [10, 994, 346, 355, 709, 132, 339, 91, 191, 554, 47, 158, 45, 48, 940, 125, 169, 560, 505, 859, 62, 940, 542, 81, 129, 845, 632, 648, 226, 519, 653, 862, 265, 389, 191, 991, 911, 965, 449, 965, 631, 36, 608, 380, 969, 771, 763, 811, 763, 648];
weights7 = [24, 13, 18, 14, 65, 96, 47, 88, 98, 55, 89, 86, 77, 17, 95, 21, 98, 37, 41, 63, 74, 11, 98, 65, 26, 83, 92, 12, 92, 12, 18, 50, 37, 14, 100, 83, 15, 92, 83, 41, 17, 23, 64, 72, 65, 22, 12, 71, 93, 58];

kp7 = KnapsackProblem(capacities7, weights7, profits7)
results7 = test(kp7, n_cases=10)

capacities8 = [469, 564, 970];
profits8 = [140, 764, 156, 269, 638, 105, 190, 129, 107, 827, 476, 457, 641, 619, 565, 948, 54, 412, 313, 336, 818, 356, 772, 413, 896, 73, 235, 988, 311, 336, 756, 762, 49, 150, 491, 955, 278, 862, 69, 956, 415, 721, 112, 264, 465, 30, 437, 311, 245, 132, 175, 338, 123, 455, 988, 77, 325, 516, 960, 816, 521, 371, 380, 765, 185, 686, 910, 853, 356, 91, 542, 96, 375, 696, 369];
weights8 = [20, 50, 64, 69, 59, 86, 33, 71, 84, 37, 87, 52, 22, 74, 63, 64, 27, 79, 32, 73, 62, 100, 100, 87, 47, 15, 79, 32, 95, 43, 11, 85, 63, 23, 50, 61, 78, 43, 10, 28, 83, 40, 17, 14, 70, 89, 21, 18, 16, 92, 96, 72, 59, 14, 26, 26, 10, 85, 31, 62, 78, 97, 81, 31, 75, 57, 53, 72, 32, 45, 59, 75, 44, 85, 34];

kp8 = KnapsackProblem(capacities8, weights8, profits8)
results8 = test(kp8, n_cases=10)

capacities9 = [562, 685, 960];
profits9 = [666, 819, 776, 951, 472, 866, 586, 982, 701, 920, 138, 604, 174, 21, 162, 439, 155, 707, 196, 890, 138, 201, 643, 188, 629, 421, 544, 601, 227, 682, 614, 538, 825, 464, 181, 156, 386, 745, 777, 769, 520, 151, 614, 562, 967, 444, 181, 296, 263, 442, 150, 487, 276, 208, 486, 934, 751, 233, 542, 277, 100, 979, 774, 11, 57, 927, 289, 915, 130, 90, 491, 481, 377, 392, 682, 893, 99, 375, 761, 486, 752, 604, 513, 806, 864, 312, 284, 95, 705, 251, 272, 699, 626, 82, 703, 714, 590, 740, 628, 651];
weights9 = [50, 48, 32, 84, 69, 76, 53, 95, 63, 99, 91, 19, 58, 57, 72, 74, 14, 18, 12, 57, 66, 42, 54, 24, 86, 53, 71, 62, 53, 19, 65, 21, 47, 53, 18, 44, 76, 51, 36, 13, 97, 56, 16, 71, 13, 45, 13, 88, 96, 67, 17, 34, 70, 98, 51, 45, 39, 68, 19, 11, 77, 17, 53, 26, 34, 19, 93, 85, 95, 12, 53, 59, 92, 74, 48, 36, 78, 34, 73, 24, 14, 75, 39, 76, 75, 31, 41, 94, 24, 29, 56, 73, 70, 65, 21, 99, 60, 51, 10, 68];

kp9 = KnapsackProblem(capacities9, weights9, profits9)
results9 = test(kp9, n_cases=10)



=== Testy dla buffer_length = 1 ===
Przypadek 1: Najlepsza wartość = 1652, czas = 1.6e-8 sekund
Przypadek 2: Najlepsza wartość = 1587, czas = 1.4e-8 sekund
Przypadek 3: Najlepsza wartość = 1582, czas = 1.3e-8 sekund
Przypadek 4: Najlepsza wartość = 1627, czas = 1.4e-8 sekund
Przypadek 5: Najlepsza wartość = 1210, czas = 1.3e-8 sekund
Przypadek 6: Najlepsza wartość = 1459, czas = 1.5e-8 sekund
Przypadek 7: Najlepsza wartość = 1550, czas = 1.3e-8 sekund
Przypadek 8: Najlepsza wartość = 1448, czas = 1.5e-8 sekund
Przypadek 9: Najlepsza wartość = 1302, czas = 1.4e-8 sekund
Przypadek 10: Najlepsza wartość = 1645, czas = 1.3e-8 sekund

Podsumowanie dla buffer_length=1:
Średnia wartość: 1506.2
Średni czas: 1.4000000000000001e-8 sekund

=== Testy dla buffer_length = 2 ===
Przypadek 1: Najlepsza wartość = 1467, czas = 1.4e-8 sekund
Przypadek 2: Najlepsza wartość = 1436, czas = 1.4e-8 sekund
Przypadek 3: Najlepsza wartość = 1427, czas = 1.3e-8 sekund
Przypadek 4: Najlepsza wartość = 1229, czas 

Dict{Int64, Vector{Tuple{Float64, Float64}}} with 3 entries:
  5 => [(21848.0, 2.2e-8), (21639.0, 2.3e-8), (25032.0, 2.8e-8), (24478.0, 2.7e…
  2 => [(22338.0, 2.0e-8), (21434.0, 2.0e-8), (21661.0, 2.0e-8), (20935.0, 1.9e…
  1 => [(20835.0, 2.4e-8), (23231.0, 2.5e-8), (25386.0, 2.3e-8), (23623.0, 2.0e…

In [44]:
using CSV
using DataFrames

function save_averages_pivoted(all_results, output_file="tabu_summary_pivoted.csv")
    df = DataFrame(
        Problem=Int[],
        AvgValue_Buffer1=Float64[], AvgTime_Buffer1=Float64[],
        AvgValue_Buffer2=Float64[], AvgTime_Buffer2=Float64[],
        AvgValue_Buffer5=Float64[], AvgTime_Buffer5=Float64[]
    )

    for (problem_idx, results_dict) in enumerate(all_results)
        avgvals = Dict(1 => (0.0, 0.0), 2 => (0.0, 0.0), 5 => (0.0, 0.0))

        for buffer_length in [1, 2, 5]
            results = results_dict[buffer_length]
            values = [res[1] for res in results]
            times = [res[2] for res in results]

            avg_value = mean(values)
            avg_time = mean(times)

            avgvals[buffer_length] = (avg_value, avg_time)
        end

        push!(df, (
            problem_idx-1,
            avgvals[1][1], avgvals[1][2],
            avgvals[2][1], avgvals[2][2],
            avgvals[5][1], avgvals[5][2]
        ))
    end

    CSV.write(output_file, df)
    println("✅ Średnie wyniki zapisane do pliku $output_file")
end

all_results = [results0, results1, results2, results3, results4, results5, results6, results7, results8, results9]

save_averages_pivoted(all_results, "tabu_summary.csv")


✅ Średnie wyniki zapisane do pliku tabu_summary.csv


## Test 2

In [23]:
using DataStructures
using Random
using Dates
using Printf

# === Parsowanie plików .dzn ===

function parse_dzn(filepath)
    capacities = []
    weights = []
    profits = []

    for line in eachline(filepath)
        line = strip(line)
        if startswith(line, "capacity")
            cap_text = strip(split(line, "=")[2], [' ', '[', ']', ';'])
            capacities = parse.(Int, split(cap_text, ","))
        elseif startswith(line, "weights")
            w_text = strip(split(line, "=")[2], [' ', '[', ']', ';'])
            weights = parse.(Int, split(w_text, ","))
        elseif startswith(line, "profits")
            p_text = strip(split(line, "=")[2], [' ', '[', ']', ';'])
            profits = parse.(Int, split(p_text, ","))
        end
    end

    return capacities, weights, profits
end

# === Definicje problemu plecakowego ===

struct KnapsackProblem
    capacities::Vector{Int}
    weights::Vector{Int}
    profits::Vector{Int}
end

function objective(p::KnapsackProblem, x)
    total = 0
    for (i, assignment) in enumerate(x)
        if assignment != 0
            total += p.profits[i]
        end
    end
    return -total
end

function is_feasible(p::KnapsackProblem, x)
    for plecak in 1:3
        total_weight = sum((p.weights[i] for i in eachindex(x) if x[i] == plecak); init=0)
        if total_weight > p.capacities[plecak]
            return false
        end
    end
    return true
end

function apply!(x, move::Tuple{Symbol,Int,Int})
    action, item_idx, new_assignment = move
    if action == :assign
        x[item_idx] = new_assignment
    elseif action == :unassign
        x[item_idx] = 0
    end
    return x
end

function invert_move(p::KnapsackProblem, move::Tuple{Symbol,Int,Int})
    action, item_idx, new_assignment = move
    if action == :assign
        return (:unassign, item_idx, 0)
    elseif action == :unassign
        return (:assign, item_idx, new_assignment)
    end
end

function possible_moves(p::KnapsackProblem, x)
    moves = Tuple{Symbol,Int,Int}[]
    for (i, assignment) in enumerate(x)
        if assignment == 0
            for plecak in 1:3
                total_weight = sum((p.weights[j] for j in eachindex(x) if x[j] == plecak); init=0) + p.weights[i]
                if total_weight <= p.capacities[plecak]
                    push!(moves, (:assign, i, plecak))
                end
            end
        else
            push!(moves, (:unassign, i, assignment))
        end
    end
    return moves
end

mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves), typeof(x0), typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end

function solve_tabu(p, s::TabuState; iteration_limit::Int=1000)
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move_idx = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                continue
            end
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            if !is_feasible(p, s.considered)
                continue
            end
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move_idx = i_move
                best_move_obj = considered_value
            end
        end
        if best_move_idx == 0
            break
        end
        apply!(s.current, moves[best_move_idx])
        push!(s.tabu_buffer, invert_move(p, moves[best_move_idx]))
        if best_move_obj < s.best_seen_obj
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    return s.best_seen
end

function random_start_solution(p::KnapsackProblem)
    x = fill(0, length(p.weights))
    for i in eachindex(p.weights)
        assign_to = rand(0:3)
        if assign_to != 0
            total_weight = sum((p.weights[j] for j in eachindex(x) if x[j] == assign_to); init=0) + p.weights[i]
            if total_weight <= p.capacities[assign_to]
                x[i] = assign_to
            end
        end
    end
    return x
end

# === Testowanie 10 problemów ===

function test_problems(filepaths)
    total_value = 0.0
    total_time = 0.0
    n_cases = length(filepaths)

    for (idx, filepath) in enumerate(filepaths)
        println("\n== Problem $idx ==")
        capacities, weights, profits = parse_dzn(filepath)
        kp = KnapsackProblem(capacities, weights, profits)

        x0 = random_start_solution(kp)
        println("Startowe przypisanie:")
        println(x0)

        st = TabuState(kp, x0; buffer_length=10)

        start_time = now()
        sol = solve_tabu(kp, st; iteration_limit=1000)
        end_time = now()

        elapsed = (end_time - start_time).value / 1e9  # sekundy
        best_value = -st.best_seen_obj

        println(@sprintf("Najlepsza wartość: %.2f", best_value))
        println(@sprintf("Czas rozwiązania: %.6f sekund", elapsed))

        total_value += best_value
        total_time += elapsed
    end

    avg_value = total_value / n_cases
    avg_time = total_time / n_cases

    println("\n=== Podsumowanie ===")
    println(@sprintf("Średnia wartość: %.2f", avg_value))
    println(@sprintf("Średni czas: %.6f sekund", avg_time))
end

# === Lista plików do wczytania ===

filepaths = [
    "generated_data/knapsack_generated_0.dzn",
    "generated_data/knapsack_generated_1.dzn",
    "generated_data/knapsack_generated_2.dzn",
    "generated_data/knapsack_generated_3.dzn",
    "generated_data/knapsack_generated_4.dzn",
    "generated_data/knapsack_generated_5.dzn",
    "generated_data/knapsack_generated_6.dzn",
    "generated_data/knapsack_generated_7.dzn",
    "generated_data/knapsack_generated_8.dzn",
    "generated_data/knapsack_generated_9.dzn"
]

# === Start testowania ===

test_problems(filepaths)



== Problem 1 ==


BoundsError: BoundsError: attempt to access 0-element Vector{Int64} at index [3]